In [86]:
from openpyxl import load_workbook
from collections import namedtuple
import pandas as pd
import re
from pathlib import Path
from pprint import pprint
import numpy as np

In [ ]:
# etsi laskentapohja -> rekkaa pohjakansio -> etsi tekstidokumentit mahdollisuuksien mukaan (vaikka käsin) -> kansioi

p = Path(r"")
files = [x for x in p.rglob('')]
len(files)

263

In [ ]:
# create id per project, the full dataset needs row identification
file_name = ""

In [29]:
wb = load_workbook(filename=file_name, data_only=True)

In [30]:
wb.sheetnames

['Tuntihinnoittelut',
 'Materiaalihinnoittelut',
 'Purkutyöt',
 'Väliseinät sisällä',
 'Maalaukset',
 'Alakatot',
 'Laatoitukset',
 'Lattiat',
 'Oviasennukset ja listoitukset',
 'Erillishankinnat',
 'Eritelysivu',
 'Koontisivu',
 'Massalista, rak. tarvikkeet ',
 'Budjetti']

In [ ]:
# tuntihinnoittelut
lines = namedtuple('item', ['kuvaus', 'tunnit'])
rows = [lines(*row[1:]) for row in wb['Tuntihinnoittelut'].iter_rows(values_only=True) if row.count(None) < 2]
pd.DataFrame(rows, index=[file_name] * len(rows), columns=['Tuote tai työ', 'kpl'],).iloc[2:, :]



In [ ]:
# materiaalihinnoittelu

line = namedtuple('item', ['POS', 'Tuote', 'Hinta', 'Kuvaus', 'kpl', 'Yht'])

a = [line(*row[:6]) for row in wb['Materiaalihinnoittelut'].iter_rows(values_only=True)]
b = [row for row in a if isinstance(row.Yht, (int, float)) and not row.Yht == 0 and not row.Tuote == 'Yhteensä']

pd.DataFrame(b, index=[file_name] * len(b)).iloc[:, [1, 4, 2, 5]]

In [ ]:
# koontisivu

line = namedtuple('item', ['selite', 'ignore', 'hinta', 'selite2'])
a = [line(*row) for row in wb['Koontisivu'].iter_rows(values_only=True) if row.count(None) < 3]
b = [x for x in a if isinstance(x.hinta, (int, float))]
c = [x for x in b if (isinstance(x.selite, str) or isinstance(x.selite2, str)) and x.selite != 'Yhteensä']

df = pd.DataFrame(c, index=[file_name] * len(c),columns=['tuote', 'ignore', 'hinta', 'tuote2']).iloc[:, [0, 2, 3]]

mask = df['tuote'].eq(0)
df.loc[mask, 'tuote'] = df.loc[mask, 'tuote2']
df = df.iloc[:, :-1]
df

In [ ]:
# purkutyöt

line = namedtuple('item', ['a', 'tuote', 'hinta'])
lines = [line(*row[:3]) for row in wb['Purkutyöt'].iter_rows(values_only=True) if row.count(None) < 6]

df = pd.DataFrame(lines, index=[file_name] * len(lines)).iloc[:, 1:].dropna(thresh=2)
df = df[df.iloc[:, 0] != 'Tuntiveloitushinta']

df

In [ ]:
# väliseinät sisällä

line = namedtuple('item', ['a', 'tuote', 'hinta'])
lines = [line(*row[:3]) for row in wb['Väliseinät sisällä'].iter_rows(values_only=True)]
df = pd.DataFrame(lines, index=[file_name] * len(lines))
df = df[((df.iloc[:, 0].astype(str).str.contains('Hinta', na=False)) |
        (df.iloc[:, 0].astype(str).str.contains('Työ', na=False))) & 
        (pd.to_numeric(df.iloc[:, 2], errors='coerce') != 0)]

df = df.iloc[:, 1:]
df.iloc[:, 0] = df.iloc[:, 0].apply(lambda x: 'työt' if x == None else x)

df

In [ ]:
# maalaukset

line = namedtuple("Line", ["työ", "hinta"])

lines = [
    line(*row[:2])
    for row in wb["Maalaukset"].iter_rows(values_only=True)
]

pos_blocks = {}
test_blocks = []
current_pos = None

for line in lines:
    # Uusi POS
    if isinstance(line.työ, str) and re.fullmatch(r"POS-\d+", line.työ.strip()):
        current_pos = line.työ.strip()
        pos_blocks[current_pos] = []
        continue

    # ei mitään ennen ekaa POS
    if current_pos is None:
        continue

    # rivien filtteröinti
    if (
        line.työ is not None
        and isinstance(line.hinta, (int, float))
        and line.hinta > 0
    ):
        pos_blocks[current_pos].append(line._asdict())
        test_blocks.append(line._asdict())

df = pd.DataFrame(test_blocks, index=[file_name] * len(test_blocks))
df

df = df[df["työ"].astype(str).str.contains("Työ|materiaalit", na=False)]

df

In [ ]:
# alakatot

line = namedtuple("Line", ["työ", "hinta"])

lines = [
    line(*row[:2])
    for row in wb["Alakatot"].iter_rows(values_only=True)
]

pos_blocks = {}
test_blocks = []
current_pos = None

for line in lines:
    # Uusi POS
    if isinstance(line.työ, str) and re.fullmatch(r"POS-\d+", line.työ.strip()):
        current_pos = line.työ.strip()
        pos_blocks[current_pos] = []
        continue

    # ei mitään ennen ekaa POS
    if current_pos is None:
        continue

    # rivien filtteröinti
    if (
        line.työ is not None
        and isinstance(line.hinta, (int, float))
        and line.hinta > 0
    ):
        pos_blocks[current_pos].append(line._asdict())
        test_blocks.append(line._asdict())

df = pd.DataFrame(test_blocks, index=[file_name] * len(test_blocks))
df = df[~df.iloc[:, 0].astype(str).str.contains('Lisä')]
df

In [ ]:
# laatoitukset

# alakatot

line = namedtuple("Line", ["työ", "hinta"])

lines = [
    line(*row[:2])
    for row in wb["Laatoitukset"].iter_rows(values_only=True)
]

pos_blocks = {}
test_blocks = []
current_pos = None

for line in lines:
    # Uusi POS
    if isinstance(line.työ, str) and re.fullmatch(r"POS-\d+", line.työ.strip()):
        current_pos = line.työ.strip()
        pos_blocks[current_pos] = []
        continue

    # ei mitään ennen ekaa POS
    if current_pos is None:
        continue

    # rivien filtteröinti
    if (
        line.työ is not None
        and isinstance(line.hinta, (int, float))
        and line.hinta > 0
    ):
        pos_blocks[current_pos].append(line._asdict())
        test_blocks.append(line._asdict())

df = pd.DataFrame(test_blocks, index=[file_name] * len(test_blocks))
df = df[~df.iloc[:, 0].astype(str).str.contains('Lisä')]
df

In [ ]:
# lattiat

line = namedtuple("Line", ["työ", "hinta"])

lines = [
    line(*row[:2])
    for row in wb["Lattiat"].iter_rows(values_only=True)
]

pos_blocks = {}
test_blocks = []
current_pos = None

for line in lines:
    # Uusi POS
    if isinstance(line.työ, str) and re.fullmatch(r"POS-\d+", line.työ.strip()):
        current_pos = line.työ.strip()
        pos_blocks[current_pos] = []
        continue

    # ei mitään ennen ekaa POS
    if current_pos is None:
        continue

    # rivien filtteröinti
    if (
        line.työ is not None
        and isinstance(line.hinta, (int, float))
        and line.hinta > 0
    ):
        pos_blocks[current_pos].append(line._asdict())
        test_blocks.append(line._asdict())

df = pd.DataFrame(test_blocks, index=[file_name] * len(test_blocks))
df = df[~df.iloc[:, 0].astype(str).str.contains('Lisä')]
df = df[df['työ'].astype(str).str.contains('materiaali|Työ')]
df

In [ ]:
# oviasennukset ja listoitukset

# alakatot

line = namedtuple("Line", ["työ", "hinta"])

lines = [
    line(*row[:2])
    for row in wb["Oviasennukset ja listoitukset"].iter_rows(values_only=True)
]

pos_blocks = {}
test_blocks = []
current_pos = None

for line in lines:
    # Uusi POS
    if isinstance(line.työ, str) and re.fullmatch(r"POS-\d+", line.työ.strip()):
        current_pos = line.työ.strip()
        pos_blocks[current_pos] = []
        continue

    # ei mitään ennen ekaa POS
    if current_pos is None:
        continue

    # rivien filtteröinti
    if (
        line.työ is not None
        and isinstance(line.hinta, (int, float))
        and line.hinta > 0
    ):
        pos_blocks[current_pos].append(line._asdict())
        test_blocks.append(line._asdict())

df = pd.DataFrame(test_blocks, index=[file_name] * len(test_blocks))
df = df[~df.iloc[:, 0].astype(str).str.contains('Lisä')]
df = df[df['työ'].astype(str).str.contains('hinta', case=False)]
df

In [ ]:
# erillishankinnat

line = namedtuple('item', ['työ', 'hinta'])

lines = [row[1:3] for row in wb['Erillishankinnat'].iter_rows(
    values_only=True
    )
   if str(row[1:3][0]) not in ('Kate-%', 'Kate-osuus')
   and row[1:3].count(None) < 1
   ][1:]

lines = [line for line in lines if line[1] > 30]

item = [x[0] for x in lines[::2]]
hinta = [x[1] for x in lines[1::2]]

lines = [x for x in zip(item, hinta)]
lines = [line(*x) for x in lines]

df = pd.DataFrame(lines, index=[file_name] * len(lines))
df_e = pd.DataFrame([{'selite': 'erillishankinnat', 'materiaalit': df['hinta'].sum(), 'työt': 0}])
df_e

In [99]:
# erittelysivu

line1 = namedtuple('item', ['selite', 'materiaalit', 'työt'])
line2 = namedtuple('item', ['Erillishankinnat', 'Hinta', 
                            'Toimittaja_urakoitsija', 'ignore1', 
                            'Toimitus', 'ignore2', 'ignore3'])

erittelyt_cols_left = []
erittelyt_cols_right = []
erillishankinnat_cols = []
kokonaishinnat = []

temp = []

for i, row in enumerate(wb['Eritelysivu'].iter_rows(values_only=True)):
    if i < 77:
        erittelyt_cols_left.append(line1(*row[:3]))
        erittelyt_cols_right.append(line1(*row[4:7]))
    if 77 <= i < 97:
        erittelyt_cols_left.append(line1(*row[:3]))
    if 97 <= i < 114:
        erillishankinnat_cols.append(line2(*row))
    if i > 114:
        if row.count(None) < 5:
            if len(temp) == 0:
                ntuple_names = list(row)
                ntuple_names = [f'id_{i}' if x is None else x for i, x in enumerate(ntuple_names)]
                ntuple_names = [x.replace('-', '_') for x in ntuple_names]
                ntuple = namedtuple('item', ntuple_names)
                temp.append(ntuple)
                continue
            if len(temp) != 0:
                out = temp.pop()
                out = out(*row)
                kokonaishinnat.append(out)


df_erillishankinnat = pd.DataFrame(erillishankinnat_cols).iloc[:, [0, 1,4]]

df_left = pd.DataFrame(erittelyt_cols_left)
df_right = pd.DataFrame(erittelyt_cols_right)
df_right.columns = df_left.columns
df_erittelyt = pd.concat([df_left, df_right])

# TODO: tarvitaanko kokonaishinnat jos on jo vastaavat erittelysivulla?
c = [a._asdict() for a in kokonaishinnat]

df = df_erittelyt[df_erittelyt['selite'].astype(str).str.contains('tuntihin|välis|ovet|hinta|tunnit', case=False)]
df[['materiaalit', 'työt']] = df[['materiaalit', 'työt']].apply(pd.to_numeric, errors='coerce')
df = df.dropna(thresh=2)

# korjaa laatoituksen otsikko
mask = df["selite"].eq("Hinta, lattiat")

if mask.sum() >= 2:
    second_idx = df.index[mask][1]
    df.loc[second_idx, "selite"] = "Hinta, laatoitukset"

df = pd.concat([df, df_e])

d = df.copy()

# fix the special material row
mask = d["selite"].eq("Hinta, materiaalit")
d.loc[mask, "materiaalit"] = d.loc[mask, "työt"]
d.loc[mask, "työt"] = 0



# category mapping

def get_category(s):
    s = str(s).lower()

    if "tuntihinnoittelut" in s or "materiaalit" in s:
        return "työmaan yleiskulut"
    if "purkutyöt" in s:
        return "purkutyöt"
    if "maalaukset" in s or "maalaus" in s:
        return "maalaukset"
    if "lattiat" in s:
        return "lattian_asennukset"
    if "laatoit" in s:
        return "laatoitukset"
    if "väliseinät" in s:
        return "väliseinät"
    if "alakatot" in s:
        return "alakatot"
    if "ovet" in s or "listat" in s:
        return "ovet_ja_listoitukset"

    return "erillishankinnat"

d["category"] = d["selite"].map(get_category)

# hour rows have hours in the "työt" column
is_hours = d["selite"].str.contains("tunnit", case=False, na=False)

d["tunnit"] = np.where(is_hours, d["työt"], 0)
d.loc[is_hours, ["materiaalit", "työt"]] = 0

out = (
    d.groupby("category", as_index=False)
     .agg({
         "materiaalit": "sum",
         "työt": "sum",
         "tunnit": "sum",
     })
)

out["yhteensa"] = out["materiaalit"] + out["työt"]

out

,category,materiaalit,työt,tunnit,yhteensa
0,alakatot,9826.250000,5823.529412,138.655462,15649.779412
1,erillishankinnat,29181.818182,0.000000,0.000000,29181.818182
2,laatoitukset,7219.617500,6516.607059,155.157311,13736.224559
3,lattian_asennukset,21981.250000,4870.588235,115.966387,26851.838235
4,maalaukset,6632.811111,10008.882353,238.306723,16641.693464
5,ovet_ja_listoitukset,11768.055556,2623.341176,62.460504,14391.396732
6,purkutyöt,5412.500000,1976.470588,47.058824,7388.970588
7,työmaan yleiskulut,5491.764706,12432.000000,0.000000,17923.764706
8,väliseinät,16107.679808,16277.176471,387.551821,32384.856279
